# AI in your hand · lessons 5 and 6: make the model

Up to now you flashed models someone else trained. Here you train one. Two ways:

- **Lesson 5:** Franta, the car, learns a trick of yours. A minute or two.
- **Lesson 6:** a robot of your own, in any language. Seconds.

First: **Runtime → Change runtime type → T4 GPU**. Lesson 6 also runs without it, only slower (a few minutes).
Then run the cells from the top. Nothing here needs the board; the board comes at the end, on your laptop.

In [ ]:
!git clone -q --depth 1 https://github.com/moudrkat/esp32-robot
%cd esp32-robot
!pip -q install tokenizers
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, lesson 6 still works")

In [ ]:
# two small helpers: write a robot's tricks to a file, and ask a trained model what it would write
import json, subprocess
def write_tricks(path, tricks):
    with open(path, "w") as f:
        for t in tricks:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
def ask(ckpt, data_dir, vocab, sentences):
    open("/tmp/ask.txt", "w").write("\n".join(sentences) + "\n")
    print(subprocess.run(["python", "-m", "research.robot.talk_test", ckpt, "/tmp/ask.txt", "--data-dir", data_dir,
                          "--vocab", str(vocab)], capture_output=True, text=True, env={**__import__("os").environ,
                          "PYTHONPATH": "src:."}).stdout)

## The language

Franta does not write Python for your laptop. He writes a small language his chip can run and check:

| does | asks |
|---|---|
| `clear() text(x, y, "hi") pixel line rect fill circle` the 128×64 screen | `distance()` cm ahead, `behind()` |
| `move(forward, sideways, turn)` `stop()` wheels | `temp()` `humidity()` thermometer |
| `arm(0..150)` `look(-90..90)` `nod(a)` servos | `sound()` microphone, `tilt()` `shaken()` |
| `tone(hz, ms)` `beep(ms)` voice | `knob()` the potentiometer, `button()` |
| `rgb(r, g, b)` `led(on)` lights, `eyes("happy")` the 8×8 face | `track()` the line sensor |
| `wait(ms)`, `for i in range(3):`, `while`, `if`, variables | |

Eyes: `open happy sad angry sleepy closed wink heart left right up down surprised dizzy`.
A trick is one program, and the ways people ask for it. The more ways, the smarter it gets.

## Lesson 5 · Franta learns a trick

Change the trick below or keep it. Then train. It starts from the finished Franta, so it is quick.

In [ ]:
write_tricks("research/robot/tricks.jsonl", [
  {"program": 'eyes("happy")\nfor i in range(3):\n    arm(150)\n    tone(784, 120)\n    arm(0)\n    tone(523, 120)\neyes("wink")\n',
   "p": ["do the robot", "do the robot dance", "dance like a robot", "robot dance!", "show me your robot moves",
         "be a robot", "act like a robot", "do your robot thing", "beep boop dance", "move like a robot",
         "can you do the robot", "robot moves please", "give me the robot", "time for the robot dance"]},
])
!./teach.sh 2>&1 | grep -E "unique pairs|DONE|done"

In [ ]:
# ask it things it never saw. Change the sentences.
ask("runs/ple-teach-s0-best.pt", "data/teach", 2048,
    ["pretend you're a robot and dance", "robot time", "drive forward 2 seconds", "can you wash the dishes"])

Try: ask it the trick in words you did **not** write. Then ask something next to it (*"robot time"*). Where does
it stop being your trick?

You can say: a model is not told what to do, it is shown. Show it fourteen ways of asking and it guesses the fifteenth.

## Lesson 6 · A robot of your own

Not a car. A plant that complains, a lamp with opinions, a cat feeder with moods. Its parts are Franta's (servo,
buzzer, screen, lights, thermometer, rangefinder), so the language above is its whole world.

The example is a **Czech** plant, on purpose: the model learns its words from your tricks, so it speaks whatever
you write. No English underneath. (Diacritics are dropped, the chip reads ASCII.) Write your own tricks in your own
language, and add one trick for what it can't do, or it will say "can't do that" in English.

In [ ]:
NAME = "kytka"
write_tricks(f"robots/{NAME}/tricks.jsonl", [
  {"program": 'if humidity() < 40:\n    eyes("sad")\n    clear()\n    text(34, 28, "zalij me!")\nelse:\n    eyes("happy")\n    clear()\n    text(28, 28, "mam se fajn")\n',
   "p": ["máš žízeň", "chceš napít", "mám tě zalít", "zalít?", "jak je ti v květináči", "potřebuješ vodu", "nemáš sucho",
         "dala by sis vodu", "máš dost vody", "žízeň?", "je čas tě zalít", "jsi suchá"]},
  {"program": 'tone(523, 200)\ntone(659, 200)\ntone(784, 400)\n',
   "p": ["zazpívej", "zazpívej mi něco", "písničku prosím", "zahraj něco", "zabrnkej", "udělej muziku", "zabzuč si",
         "dej mi písničku", "hraj", "la la la", "zpívej pro mě", "čas na muziku"]},
  {"program": 'for i in range(3):\n    arm(150)\n    wait(300)\n    arm(0)\n    wait(300)\n',
   "p": ["zamávej", "zamávej mi", "pozdrav", "ahoj!", "zamávej listem", "čau", "pozdrav mě", "mávni", "nazdar",
         "dobré ráno", "řekni ahoj", "pa pa"]},
  {"program": 'rgb(0, 0, 0)\neyes("sleepy")\nclear()\ntext(52, 28, "zzz")\n',
   "p": ["dobrou noc", "jdi spát", "spinkej", "hajat", "zhasni", "čas jít spát", "dej si šlofíka", "dobrou",
         "odpočívej", "sladké sny", "do postele", "vypni se"]},
  {"program": 'eyes("sad")\nclear()\ntext(28, 28, "to neumim")\n',
   "p": ["umyj okno", "uvař kafe", "vyvenči psa", "jeď do obchodu", "udělej mi čaj", "ukliď pokoj", "vyper prádlo",
         "zavolej mamince", "kup rohlíky", "ustel postel", "nakrm kočku", "dones noviny", "vysávej", "uvař oběd"]},
])
!./mine.sh robots/{NAME} 2>&1 | grep -E "unique pairs|total=|DONE|done"

In [ ]:
ask(f"runs/ple-mine-{NAME}-s0-best.pt", f"robots/{NAME}/data", 512,
    ["chtěla bys zalít", "zazpívej písničku", "ahoj kytko", "už spinkej", "umyj nádobí", "jeď dopředu"])

Try: change one wording and train again (it takes seconds). Then the bold one: give it a trick with **one**
wording only, and ask it in another. Count how many wordings it needs before it guesses.

You can say: the model has no idea it is a plant. It knows the sentences it was shown and the programs that went
with them. Everything else is a guess, and "to neumim" is a guess too: you taught it that one.

## To the board

Download the model, unzip it into the `esp32-robot` folder on your laptop, and flash:

```
MODEL=robots/kytka/model ./flash.sh        # lesson 6 (or MODEL=model-teach for lesson 5)
python talk.py
```

`MODEL=0 ./flash.sh` gives you Franta back.

In [ ]:
import shutil; from google.colab import files
folder = f"robots/{NAME}/model"          # lesson 5: "model-teach"
shutil.make_archive("my-model", "zip", ".", folder)
files.download("my-model.zip")